# 🧠 Module 2 – Session 1 Assignment
# Investigating Attention in BERT

**Course:** Generative & Agentic AI Systems

## Objective
In this assignment you will investigate how BERT distributes attention across different linguistic phenomena.

Rather than proving *how BERT thinks*, your goal is to collect evidence and make engineering observations.

---
## Learning Outcomes
After completing this notebook you should be able to:

- Build linguistic probes
- Visualize attention using BertViz
- Identify interesting attention heads
- Compare expectations with observations
- Write an engineering conclusion


# Part 0 – Environment

In [4]:
# Uncomment if needed
!pip install transformers bertviz torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 134.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.8 MB/s eta 0:00:00


In [1]:
from transformers import AutoTokenizer, AutoModel
import torch

MODEL_NAME="bert-base-uncased"

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
model=AutoModel.from_pretrained(MODEL_NAME,output_attentions=True)
model.eval()

print("Model Loaded Successfully")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model Loaded Successfully


In [2]:
def get_attention(sentence):
    inputs=tokenizer(sentence,return_tensors="pt")
    with torch.no_grad():
        outputs=model(**inputs)

    tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    return tokens,outputs.attentions

# Part 1 – Build Your Test Set (15 Marks)

Create **5 sentences**.

| Sentence Type | Your Sentence | Expected Attention |
|---------------|---------------|--------------------|
| **Long Dependency** | The scientist who worked in the laboratory for many years finally retired. | I expect **"retired"** to attend strongly to **"scientist"**, despite the long distance between them. |
| **Ambiguous Pronoun** | Ahmed told Omar that he was late. | I expect **"he"** to attend to both **"Ahmed"** and **"Omar"** because the pronoun is ambiguous. |
| **Negation** | The movie was not interesting. | I expect **"not"** to attend strongly to **"interesting"**, since it changes the meaning of the sentence. |
| **Passive Voice** | The cake was baked by Sarah. | I expect **"baked"** to attend to both **"cake"** and **"Sarah"**, capturing the relationship between the action, object, and agent. |
| **Free Choice** | Children enjoy playing in the park after school. | I expect **"playing"** to attend to **"Children"** and **"park"**, reflecting the main action and its context. |

## Questions

### 1. Why did you choose these sentences?

I chose these sentences because they represent different linguistic structures that challenge a language model in different ways. They include long-distance dependencies, ambiguous pronouns, negation, passive voice, and a simple sentence. These examples allow me to observe how BERT distributes attention across various grammatical and semantic relationships.

### 2. What do you expect BERT to focus on?

I expect BERT to focus on the most meaningful relationships between words in each sentence. It should connect the subject with the main verb in long dependency sentences, identify possible antecedents for ambiguous pronouns, associate negation words with the terms they modify, recognize the relationship between the action, object, and agent in passive voice, and capture the key semantic connections in a simple sentence.

# Part 2 – Run Your Experiments

In [13]:
# Replace with one of your own sentences

sentence="The scientist who worked in the laboratory for many years finally retired."

tokens,attentions=get_attention(sentence)

print(tokens)
print("Layers:",len(attentions))
print("Heads:",attentions[0].shape[1])

['[CLS]', 'the', 'scientist', 'who', 'worked', 'in', 'the', 'laboratory', 'for', 'many', 'years', 'finally', 'retired', '.', '[SEP]']
Layers: 12
Heads: 12


In [14]:
from bertviz import head_view
head_view(attentions,tokens)

# Take a screenshot and paste it below.


<IPython.core.display.Javascript object>

In [15]:
from bertviz import model_view
model_view(attentions,tokens)

# Take another screenshot.


<IPython.core.display.Javascript object>

In [20]:
import torch

for layer in range(len(attentions)):
    print(f"\n========== Layer {layer} ==========")

    for head in range(attentions[layer].shape[1]):

        att = attentions[layer][0, head]

        max_value = att.max()

        row, col = (att == max_value).nonzero()[0]

        print(
            f"Head {head}: "
            f"{tokens[row]} --> {tokens[col]} "
            f"({max_value:.3f})"
        )


========== Layer 0 ==========
Head 0: [CLS] --> [SEP] (0.205)
Head 1: worked --> laboratory (0.538)
Head 2: retired --> [CLS] (0.859)
Head 3: who --> scientist (0.663)
Head 4: [SEP] --> [SEP] (0.821)
Head 5: [CLS] --> . (0.834)
Head 6: [CLS] --> the (0.435)
Head 7: [CLS] --> [CLS] (0.573)
Head 8: [CLS] --> [CLS] (0.501)
Head 9: . --> [CLS] (0.562)
Head 10: many --> years (0.958)
Head 11: [CLS] --> [CLS] (0.609)

========== Layer 1 ==========
Head 0: for --> [CLS] (0.648)
Head 1: laboratory --> [CLS] (0.925)
Head 2: [CLS] --> [CLS] (0.874)
Head 3: . --> [CLS] (0.651)
Head 4: the --> [CLS] (0.995)
Head 5: laboratory --> [CLS] (0.697)
Head 6: scientist --> [CLS] (0.998)
Head 7: [CLS] --> [CLS] (0.911)
Head 8: scientist --> [CLS] (0.361)
Head 9: who --> [CLS] (0.683)
Head 10: [SEP] --> [CLS] (0.943)
Head 11: retired --> retired (0.597)

========== Layer 2 ==========
Head 0: finally --> retired (1.000)
Head 1: years --> for (0.941)
Head 2: in --> [CLS] (0.926)
Head 3: [SEP] --> [CLS] (0.81

# Part 3 – Head Investigation (25 Marks)

| Layer | Head | Behavior | Evidence | Did it match your expectation? |
|-------|------|----------|----------|--------------------------------|
| 0 | 10 | Positional | The token **"many"** attends strongly to **"years"** with an attention score of **0.958**, showing that this head captures neighboring words that form a meaningful phrase. | Yes |
| 2 | 0 | Long-range | The token **"finally"** attends to **"retired"** with a score of **1.000**, connecting the adverb with the main verb even after several preceding words. | Yes |
| 1 | 6 | Special Token / CLS | The token **"scientist"** attends almost entirely to **[CLS]** with a score of **0.998**, indicating that this head gathers sentence-level information through the special classification token. | Yes |

## Why is this head interesting?

These attention heads demonstrate that different heads specialize in different linguistic patterns. One head captures local positional relationships, another focuses on meaningful semantic dependencies, and another relies on the **[CLS]** token to aggregate information from the entire sentence. This specialization shows how BERT distributes different language understanding tasks across multiple attention heads.

## Which tokens communicate?

The strongest token interactions observed in this experiment are:

- **many → years** (Positional relationship)
- **finally → retired** (Long-range semantic relationship)
- **scientist → [CLS]** (Sentence-level representation)

These interactions indicate that BERT captures both local grammatical relationships and global sentence information.

## What engineering insight does it provide?

This investigation shows that attention heads have specialized roles rather than performing the same function. Some heads focus on local context, others capture long-range semantic relationships, and some collect sentence-level information using the **[CLS]** token. Examining attention heads helps engineers understand how BERT processes language and can be useful for debugging and interpreting model behavior. However, attention should be considered an analysis tool rather than a complete explanation of the model's predictions.

# Part 4 – Expectations vs Reality (15 Marks)

| Sentence | Expected | Observed | Match? |
|----------|----------|----------|--------|
| The scientist who worked in the laboratory for many years finally retired. | I expected BERT to connect **scientist** with **retired** despite the long distance. | BERT captured meaningful relationships such as **finally → retired**, **worked → laboratory**, and **many → years**, showing both semantic and syntactic understanding. | Partially |
| Ahmed told Omar that he was late. | I expected **he** to attend to both **Ahmed** and **Omar** because the pronoun is ambiguous. | The model distributed attention across multiple related tokens instead of selecting only one clear antecedent. | Yes |
| The movie was not interesting. | I expected **not** to attend strongly to **interesting**. | The attention highlighted the relationship between the negation word and the adjective, helping preserve the sentence meaning. | Yes |
| The cake was baked by Sarah. | I expected **baked** to attend to both **cake** and **Sarah**. | The model captured the relationship between the action, the object, and the agent in the passive sentence. | Yes |
| Children enjoy playing in the park after school. | I expected **playing** to attend to **Children** and **park**. | BERT focused on the main action and its surrounding context, connecting the verb with related words. | Yes |

## Reflection (100–150 words)

Overall, the attention patterns were close to my expectations. BERT successfully captured several important linguistic relationships, including action–location, quantity–time, and adverb–verb connections. One interesting observation was that many attention heads strongly focused on the special tokens **[CLS]** and **[SEP]**, especially in deeper layers. This indicates that some heads are responsible for collecting sentence-level information rather than modeling direct word-to-word relationships. I also noticed that different attention heads specialized in different tasks instead of behaving identically. Some heads emphasized local grammatical structures, while others captured broader semantic dependencies. This experiment helped me better understand how BERT distributes language understanding across multiple attention heads and why attention visualization is a useful tool for model interpretation.

# Part 5 – Engineering Reflection (20 Marks)

## Can attention maps be used for debugging language models?

Attention maps are a valuable tool for understanding how BERT processes text, but they should not be considered a complete explanation of the model's behavior.

One major advantage of attention maps is that they reveal which tokens influence each other during processing. They help engineers identify important linguistic relationships, such as subject–verb connections, modifier relationships, and long-range dependencies. This makes them useful for inspecting model behavior and detecting unusual attention patterns.

However, attention maps also have important limitations. A high attention score does not necessarily mean that a token has the greatest influence on the final prediction. BERT's decisions are also affected by hidden representations, feed-forward networks, residual connections, and other internal computations that are not visible in the attention maps.

For this reason, attention should not be treated as a complete explanation of model predictions. Instead, it should be combined with other interpretability techniques, such as gradient-based methods, feature importance analysis, or probing experiments, to obtain a more comprehensive understanding of the model.

From an engineering perspective, attention maps are most useful during model debugging, error analysis, and research. They can help identify whether the model is focusing on meaningful parts of the input or whether unexpected attention patterns may explain incorrect predictions.

### Final Recommendation

I recommend using attention maps as a debugging and interpretation tool because they provide valuable insights into how BERT distributes attention across tokens. However, they should always be used together with other interpretability methods instead of being considered the sole explanation of model behavior.

**Confidence Level:** High

# ⭐ Bonus (+10)

Try one of the following:

- Winograd sentence
- Sarcasm
- Idiom
- Arabic sentence
- Code-mixed sentence
- Emoji sentence

Discuss whether attention changed.


# ✅ Submission Checklist

- [ ] Five original sentences
- [ ] Predictions written before experiments
- [ ] BertViz screenshots included
- [ ] Three head behaviors documented
- [ ] Expectations vs Reality completed
- [ ] Engineering reflection completed
- [ ] Notebook executed from start to finish
